# Architecture Walkthrough: Parameter Golf Baseline

This notebook traces through every component of the baseline GPT model used in the OpenAI Parameter Golf challenge.

**Goal:** Understand what each piece does, why it's there, and what the numbers look like.

## The Architecture at a Glance

```
Input tokens (1024 vocab)
    │
    ▼
Token Embedding + RMSNorm
    │
    ▼
┌─ Encoder (layers 0-3) ─────────────────────┐
│  Block = Attention + MLP with residuals     │
│  Each block saves a 'skip' tensor           │
└─────────────────────────────────────────────┘
    │ + skip connections (reversed)
    ▼
┌─ Decoder (layers 4-8) ─────────────────────┐
│  Block = Attention + MLP with residuals     │
│  Each block receives a skip from encoder    │
└─────────────────────────────────────────────┘
    │
    ▼
Final RMSNorm → Logits (via tied embedding) → Softcap
```

In [ ]:
import sys
sys.path.insert(0, '../parameter-golf')

import numpy as np
import mlx.core as mx
import mlx.nn as nn
from mlx.utils import tree_flatten

# Import the model components from the baseline
from train_gpt_mlx import (
    Hyperparameters, GPT, Block, CausalSelfAttention, MLP,
    CastedLinear, RMSNormNoWeight, rms_norm, zeropower_newtonschulz5
)

## 1. Model Configuration

Let's look at the default hyperparameters and count parameters.

In [ ]:
args = Hyperparameters()

print("=== Model Configuration ===")
print(f"Vocab size:      {args.vocab_size}")
print(f"Num layers:      {args.num_layers}")
print(f"Model dim:       {args.model_dim}")
print(f"Num heads:       {args.num_heads}")
print(f"Num KV heads:    {args.num_kv_heads}  (GQA: {args.num_heads // args.num_kv_heads}x sharing)")
print(f"Head dim:        {args.model_dim // args.num_heads}")
print(f"MLP multiplier:  {args.mlp_mult}x  (hidden={args.model_dim * args.mlp_mult})")
print(f"Sequence length: {args.train_seq_len}")
print(f"Tied embeddings: {args.tie_embeddings}")
print(f"Logit softcap:   {args.logit_softcap}")
print(f"RoPE base:       {args.rope_base}")

In [ ]:
# Build the model
mx.random.seed(1337)
model = GPT(
    vocab_size=args.vocab_size,
    num_layers=args.num_layers,
    dim=args.model_dim,
    num_heads=args.num_heads,
    num_kv_heads=args.num_kv_heads,
    mlp_mult=args.mlp_mult,
    logit_chunk_tokens=args.logit_chunk_tokens,
    logit_softcap=args.logit_softcap,
    rope_base=args.rope_base,
    tied_embed_init_std=args.tied_embed_init_std,
    qk_gain_init=args.qk_gain_init,
)

# Count parameters by component
params = dict(tree_flatten(model.parameters()))
total = 0
print("\n=== Parameter Breakdown ===")
for name, p in sorted(params.items()):
    count = int(np.prod(p.shape))
    total += count
    print(f"{name:50s} {str(p.shape):20s} {count:>10,} params  ({p.dtype})")
print(f"{'TOTAL':50s} {'':20s} {total:>10,} params")
print(f"\nAt float32: {total * 4 / 1e6:.2f} MB")
print(f"At float16: {total * 2 / 1e6:.2f} MB")
print(f"At int8:    {total * 1 / 1e6:.2f} MB (before compression)")

## 2. Token Embedding + RMSNorm

The first step: map each token ID to a 512-dimensional vector, then normalize it.

**Why RMSNorm instead of LayerNorm?**
- RMSNorm skips the mean-centering step — it only normalizes by the root-mean-square
- Cheaper to compute, and empirically works just as well for transformers
- No learnable parameters here (no weight/bias) — keeps the param budget tiny

**Why is the vocab so small (1024)?**
- The embedding table is `vocab_size × model_dim` = 1024 × 512 = 524,288 params
- With tied embeddings (same matrix for input and output), this is shared
- A larger vocab would eat into the 16MB budget quickly

In [ ]:
# Simulate the first step: embedding + normalization
fake_tokens = mx.array([[42, 100, 500, 7, 999]], dtype=mx.int32)  # 1 sequence, 5 tokens

# Step 1: Look up embeddings
raw_embed = model.tok_emb(fake_tokens)
print(f"Raw embedding shape: {raw_embed.shape}")
print(f"Raw embedding dtype: {raw_embed.dtype}")
print(f"Raw embedding norm (per token): {mx.sqrt(mx.sum(raw_embed * raw_embed, axis=-1))}")

# Step 2: RMSNorm
normed = rms_norm(raw_embed.astype(mx.bfloat16))
print(f"\nAfter RMSNorm shape: {normed.shape}")
print(f"After RMSNorm norm (per token): {mx.sqrt(mx.sum(normed.astype(mx.float32) * normed.astype(mx.float32), axis=-1))}")
print("\nNotice: RMSNorm makes each token vector have roughly unit RMS (sqrt(dim) ≈ 22.6)")

## 3. Attention: GQA + RoPE

### Grouped Query Attention (GQA)
Standard multi-head attention has separate Q, K, V projections for each head.
GQA shares K and V across groups of Q heads:
- 8 query heads, 4 KV heads → each KV head serves 2 query heads
- Saves parameters: K and V projections are half the size

### Rotary Position Embeddings (RoPE)
Instead of adding position embeddings to token vectors, RoPE encodes position by *rotating* the query and key vectors:
- Each pair of dimensions gets rotated by an angle proportional to position
- The dot product `q·k` naturally encodes *relative* position
- No extra parameters needed!

### QK Normalization
Before applying RoPE, Q and K are normalized with RMSNorm, then Q is scaled by a learnable `q_gain` per head. This stabilizes training.

In [ ]:
# Look at the attention module
attn = model.blocks[0].attn

print("=== Attention Configuration ===")
print(f"Num Q heads:   {attn.num_heads}")
print(f"Num KV heads:  {attn.num_kv_heads}")
print(f"Head dim:      {attn.head_dim}")
print(f"Q projection:  {attn.c_q.weight.shape} ({int(np.prod(attn.c_q.weight.shape)):,} params)")
print(f"K projection:  {attn.c_k.weight.shape} ({int(np.prod(attn.c_k.weight.shape)):,} params)")
print(f"V projection:  {attn.c_v.weight.shape} ({int(np.prod(attn.c_v.weight.shape)):,} params)")
print(f"Out projection:{attn.proj.weight.shape} ({int(np.prod(attn.proj.weight.shape)):,} params)")
print(f"\nQ+K+V+Out = {sum(int(np.prod(x.shape)) for x in [attn.c_q.weight, attn.c_k.weight, attn.c_v.weight, attn.proj.weight]):,} params")
print(f"\nWith full MHA (8 KV heads): Q+K+V+Out would be {512*512*4:,} params")
print(f"GQA savings: {(512*512*4 - sum(int(np.prod(x.shape)) for x in [attn.c_q.weight, attn.c_k.weight, attn.c_v.weight, attn.proj.weight])):,} params saved")

In [ ]:
# Trace through attention on a small input
x = rms_norm(model.tok_emb(fake_tokens).astype(mx.bfloat16))
bsz, seqlen, dim = x.shape

# Project to Q, K, V
q = attn.c_q(x).reshape(bsz, seqlen, attn.num_heads, attn.head_dim)
k = attn.c_k(x).reshape(bsz, seqlen, attn.num_kv_heads, attn.head_dim)
v = attn.c_v(x).reshape(bsz, seqlen, attn.num_kv_heads, attn.head_dim)

print(f"Q shape: {q.shape}  (batch, seq, heads, head_dim)")
print(f"K shape: {k.shape}  (batch, seq, kv_heads, head_dim)")
print(f"V shape: {v.shape}  (batch, seq, kv_heads, head_dim)")
print(f"\nNote: K and V have 4 heads (not 8). During attention, each KV head is shared by 2 Q heads.")

## 4. MLP: ReLU²

The MLP in each block uses a fun activation: **ReLU squared**.

```python
def forward(x):
    x = relu(fc(x))     # Standard ReLU
    return proj(x * x)   # Square it!
```

Why square it?
- Creates **sparser** activations (values < 1 get even smaller, values > 1 get amplified)
- Empirically works well for small models
- Computationally cheap (no extra parameters vs SiLU/GELU)

In [ ]:
mlp = model.blocks[0].mlp
print(f"MLP fc (up):   {mlp.fc.weight.shape}  (dim → {args.model_dim * args.mlp_mult})")
print(f"MLP proj (down): {mlp.proj.weight.shape}  ({args.model_dim * args.mlp_mult} → dim)")
print(f"MLP params: {sum(int(np.prod(w.shape)) for _, w in tree_flatten(mlp.parameters())):,}")

# Visualize the sparsity of ReLU²
test_input = mx.random.normal((1, 10, args.model_dim))
after_fc = nn.relu(mlp.fc(test_input))
after_relu_sq = after_fc * after_fc

print(f"\n% zeros after ReLU:  {float(mx.mean(after_fc == 0)) * 100:.1f}%")
print(f"% near-zero after ReLU² (<0.01): {float(mx.mean(mx.abs(after_relu_sq) < 0.01)) * 100:.1f}%")
print("\nReLU² makes ~half the activations zero and pushes small values even smaller → sparse!")

## 5. U-Net Skip Connections

This is one of the more unusual features. The model splits its layers into:
- **Encoder** (first half): saves intermediate outputs as 'skips'
- **Decoder** (second half): adds those skips back in (reversed order)

This is inspired by U-Net from image segmentation — it lets the decoder access both high-level (deep) and low-level (shallow) features.

```
Layer 0 (enc) ──save──────────────────────add──► Layer 8 (dec)
Layer 1 (enc) ──save───────────────add──►        Layer 7 (dec)
Layer 2 (enc) ──save────────add──►               Layer 6 (dec)
Layer 3 (enc) ──save──add──►                     Layer 5 (dec)
                    Layer 4 (dec, no skip — odd layer count)
```

Each skip connection has a learnable weight per dimension.

In [ ]:
print(f"Total layers: {args.num_layers}")
print(f"Encoder layers: {model.num_encoder_layers} (layers 0-{model.num_encoder_layers - 1})")
print(f"Decoder layers: {model.num_decoder_layers} (layers {model.num_encoder_layers}-{args.num_layers - 1})")
print(f"Skip connections: {model.num_skip_weights}")
print(f"Skip weights shape: {model.skip_weights.shape}")
print(f"\nSkip weights (initialized to 1.0):")
print(f"Mean: {float(mx.mean(model.skip_weights)):.4f}")
print(f"This means skips start with full strength and the model learns to modulate them.")

## 6. The Block: Putting It Together

Each transformer block does:
```
x = mix[0] * x + mix[1] * x0       # Residual mixing (x0 is the original embedding)
x = x + attn_scale * Attention(RMSNorm(x))   # Attention with scaled residual
x = x + mlp_scale * MLP(RMSNorm(x))          # MLP with scaled residual
```

Notable details:
- `resid_mix`: lets each block blend between the running hidden state and the original embeddings
- `attn_scale` and `mlp_scale`: learnable per-dimension scaling of residual contributions
- Both are initialized to 1.0 (identity), so the model starts as a standard residual network

In [ ]:
block = model.blocks[0]
print("=== Block 0 Learnable Scales ===")
print(f"attn_scale shape: {block.attn_scale.shape}, mean: {float(mx.mean(block.attn_scale)):.4f}")
print(f"mlp_scale shape:  {block.mlp_scale.shape}, mean: {float(mx.mean(block.mlp_scale)):.4f}")
print(f"resid_mix shape:  {block.resid_mix.shape}")
print(f"  mix[0] (current x weight): mean={float(mx.mean(block.resid_mix[0])):.4f}")
print(f"  mix[1] (original x0 weight): mean={float(mx.mean(block.resid_mix[1])):.4f}")

# Count total params per block
block_params = sum(int(np.prod(p.shape)) for _, p in tree_flatten(block.parameters()))
print(f"\nTotal params per block: {block_params:,}")
print(f"Total params in {args.num_layers} blocks: {block_params * args.num_layers:,}")

## 7. Logit Softcap

The final logits are passed through a soft capping function:
```python
logits = 30.0 * tanh(logits / 30.0)
```

This prevents any logit from exceeding ±30, which:
- Stabilizes training (prevents extreme confidence)
- Acts as a form of label smoothing
- Used by Gemma/Gemini models

In [ ]:
# Demonstrate softcap behavior
test_logits = mx.array([-50, -30, -10, 0, 10, 30, 50], dtype=mx.float32)
capped = model.softcap(test_logits)

print("Softcap(30.0) behavior:")
print(f"{'Input':>10s} → {'Output':>10s}")
for i in range(len(test_logits)):
    print(f"{float(test_logits[i]):>10.1f} → {float(capped[i]):>10.4f}")
print(f"\nValues near 0 pass through almost unchanged.")
print(f"Values beyond ±30 get squished toward ±30.")

## 8. Full Forward Pass

Let's trace a complete forward pass and see the shapes at each stage.

In [ ]:
# Create a small batch of random tokens
batch_size = 2
seq_len = 16
tokens = mx.random.randint(0, args.vocab_size, (batch_size, seq_len))

print(f"Input tokens: {tokens.shape}")

# Forward pass with intermediate prints
x = rms_norm(model.tok_emb(tokens).astype(mx.bfloat16))
print(f"After embedding + RMSNorm: {x.shape} ({x.dtype})")

x0 = x  # Save for residual mixing
skips = []

# Encoder
for i in range(model.num_encoder_layers):
    x = model.blocks[i](x, x0)
    skips.append(x)
    print(f"After encoder layer {i}: {x.shape}")

# Decoder
for i in range(model.num_decoder_layers):
    if skips:
        skip = skips.pop()
        x = x + model.skip_weights[i].astype(x.dtype)[None, None, :] * skip
        print(f"After skip connection {i}: {x.shape}")
    x = model.blocks[model.num_encoder_layers + i](x, x0)
    print(f"After decoder layer {model.num_encoder_layers + i}: {x.shape}")

x = model.final_norm(x)
print(f"After final norm: {x.shape}")

# Tied embedding projection for logits
logits = x.reshape(-1, model.tok_emb.weight.shape[1]) @ model.tok_emb.weight.astype(x.dtype).T
logits = model.softcap(logits)
print(f"Logits: {logits.shape} (batch*seq, vocab_size)")

## 9. Size Budget Analysis

The challenge has a strict 16MB (16,000,000 bytes) limit for code + compressed weights.
Let's see how the baseline uses this budget.

In [ ]:
import zlib
import pickle
from train_gpt_mlx import quantize_state_dict_int8

flat_state = {k: v for k, v in tree_flatten(model.state)}
quant_obj, stats = quantize_state_dict_int8(flat_state)

quant_raw = pickle.dumps(quant_obj, protocol=pickle.HIGHEST_PROTOCOL)
quant_blob = zlib.compress(quant_raw, level=9)

code_bytes = len(open('../parameter-golf/train_gpt_mlx.py', 'r').read().encode('utf-8'))
model_bytes = len(quant_blob)
total_bytes = code_bytes + model_bytes
budget = 16_000_000

print("=== Size Budget ===")
print(f"Code size:            {code_bytes:>12,} bytes ({code_bytes/budget*100:.1f}%)")
print(f"Model (int8+zlib):    {model_bytes:>12,} bytes ({model_bytes/budget*100:.1f}%)")
print(f"Total:                {total_bytes:>12,} bytes ({total_bytes/budget*100:.1f}%)")
print(f"Budget:               {budget:>12,} bytes")
print(f"Remaining:            {budget - total_bytes:>12,} bytes ({(budget-total_bytes)/budget*100:.1f}%)")
print(f"\n=== Quantization Stats ===")
for k, v in stats.items():
    print(f"{k}: {v:,}")
print(f"\nCompression ratio: {stats['baseline_tensor_bytes'] / model_bytes:.2f}x")

## Key Takeaways

1. **The model is tiny** — ~3.3M parameters, mostly in attention and MLP projections
2. **GQA saves params** — 4 KV heads instead of 8 saves ~25% on K/V projections
3. **Tied embeddings** — input and output share the same 1024×512 matrix
4. **U-Net skips** — encoder-decoder with skip connections, unusual for language models
5. **ReLU²** — cheap, sparse activation that works well at this scale
6. **Int8 + zlib** — baseline quantization gets decent compression

## What the Winners Changed

The top submissions found ways to fit more capacity into the same 16MB:
- **Int6/Int5 quantization** — more aggressive compression → room for more params
- **3x MLP** — bigger hidden layer (1536 instead of 1024)
- **10-11 layers** — more depth
- **BigramHash** — hash table for bigram predictions
- **zstd** — better compression than zlib
- **SWA** — smoother weights that compress better

Next notebook: **02-quantization-deep-dive.ipynb** — Understanding how int8/int6/int5 quantization works and why it matters so much.